# Oscar winners for the AWARD_Movie table
Written by: Capleton

This notebook makes a clean CSV file that fits the table AWARD_Movie in our PostgreSQL database. First the Oscar data is cleaned, then the foreign key movie_id is filled with the IDs from the movie table, because it is a NOT NULL foreign key.

## STEP 1: import os and pandas
We need pandas to prepare the data before it goes into the database, and os to save the file for the import in pgAdmin.

In [ ]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## STEP 2: read the files
The original file stays unchanged. If something goes wrong in the database, we can always build the table again from the full data.

We need the award file and the movie table it points to. The IDs in movie are the ones in the database, so the awards must use exactly these numbers.

In [ ]:
folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data"
clean_folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\clean"

df = pd.read_csv(os.path.join(folder, "oscarAward.csv"))
movie = pd.read_csv(os.path.join(clean_folder, "movie.csv"))

print("Rows and columns:", df.shape)
print(df.head())

## STEP 3: check the movie table
Before we use the movie table, we check what is in it. movie_id must be unique, otherwise one award could be linked to two movies. We also check that title and RelDate are there, because the match key is made from them.

In [ ]:
print(movie[["movie_id", "title", "RelDate"]].head())

print("Rows in movie:", len(movie))
print("Check, movie_id is unique:", movie["movie_id"].is_unique)
print("Check, title and RelDate are there:", "title" in movie.columns and "RelDate" in movie.columns)

## STEP 4: keep only the winners with a film
The database should only hold data that answers our research questions. An award without a film can never be linked to a row in MOVIE, so it has no place in a table with a foreign key to MOVIE.

In [ ]:
winners = df[df["winner"] == True]
winners = winners.dropna(subset=["film"])

print("Rows before:", len(df))
print("Rows after:", len(winners))

## STEP 5: keep only the movie awards
In our ERD every type of award has its own table. AWARD_Movie holds the Oscars for the film itself, for example Best Picture, Cinematography or Sound. The directing and acting Oscars go to AWARD_Director and AWARD_Actor, so every Oscar is stored in exactly one table and nothing is stored twice.

In [ ]:
is_director = winners["category"].str.startswith("DIRECTING")
is_actor = winners["category"].str.startswith(("ACTOR", "ACTRESS"))
award_movie = winners[~is_director & ~is_actor].copy()

print("Rows after:", len(award_movie))
print("Number of categories:", award_movie["category"].nunique())

## STEP 6: rename the columns
The column names in the CSV must be the same as in the table in PostgreSQL. Otherwise pgAdmin does not know which value goes into which column.

In [ ]:
award_movie = award_movie.rename(columns={"name": "name_nominee", "film": "film_title"})

print(list(award_movie.columns))

## STEP 7: clean the film titles and make the match key
Some old titles in the source data contain a ;. It is a leftover and not part of the real title, so we remove it to keep the data in the database clean.

film_title_clean is the title in small letters, a hyphen becomes a space, dots are removed and double spaces become one. film_title stays as it is, so the real title with capitals is still stored.

match_key is film_title_clean plus the year of the film, for example no country for old men_2007. The database needs it to fill the foreign key movie_id: the Oscar data has no movie_id, so we look up this key in MOVIE. We add the year because different films can have the same title, like a remake. The year of the film is year_ceremony minus 1, because the Oscars are given the year after the film came out. MOVIE makes the same key from its title and the year of its release date.

In [ ]:
award_movie["film_title"] = award_movie["film_title"].str.replace(";", "").str.strip()
award_movie["film_title_clean"] = award_movie["film_title"].str.lower().str.replace("-", " ", regex=False).str.replace(".", "", regex=False).str.split().str.join(" ")

award_movie["match_key"] = award_movie["film_title_clean"] + "_" + (award_movie["year_ceremony"] - 1).astype(str)

print(award_movie[["film_title", "film_title_clean", "match_key"]].head())
print("Titles with ; left:", award_movie["film_title"].str.contains(";").sum())

## STEP 8: sort the table
A fixed order gives every award the same movie_award_id every time. If the numbers change, the rows in the database no longer match the file.

In [ ]:
award_movie = award_movie.sort_values(["year_ceremony", "category", "film_title"]).reset_index(drop=True)

print(award_movie[["year_ceremony", "category", "name_nominee", "film_title", "film_title_clean"]].head())

## STEP 9: give every award a movie_award_id
movie_award_id is the primary key of AWARD_Movie. The database needs it to tell every row apart, so it must be unique and can never be empty.

In [ ]:
award_movie["movie_award_id"] = award_movie.index + 1

print("First movie_award_id:", award_movie["movie_award_id"].min())
print("Last movie_award_id:", award_movie["movie_award_id"].max())
print("All unique:", award_movie["movie_award_id"].is_unique)

## STEP 10: link movie_id
movie_id is found by the match key (title plus year). The award file already has the match key, so the movie table gets the same key, with the same cleaning (small letters, no hyphens, no dots, no double spaces). Otherwise a small difference in writing would break the link.

Some different movies have the same title and year. For those we cannot know which movie is meant, so they are left out of the lookup. Linking an award to the wrong movie would be worse than no link.

In [ ]:
movie["match_key"] = movie["title"].str.lower().str.replace("-", " ").str.replace(".", "").str.split().str.join(" ") + "_" + movie["RelDate"].str[:4]
movie_keys = movie.drop_duplicates(subset="match_key", keep=False)

award_movie = award_movie.merge(movie_keys[["movie_id", "match_key"]], on="match_key", how="left")

print("Awards with a movie:", award_movie["movie_id"].notna().sum())
print("Awards without a movie:", award_movie["movie_id"].isna().sum())

## STEP 11: check if the links are correct
A link can exist and still be wrong. So we put the title from the movie table next to the title from the Oscar data. If they are the same, the award points to the right movie in the database.

In [ ]:
check = award_movie.dropna(subset=["movie_id"]).merge(movie[["movie_id", "title", "RelDate"]], on="movie_id")

print(check[["film_title", "title", "year_ceremony", "RelDate"]].head(10))

## STEP 12: remove awards without a link
An award for a movie that is not in our database cannot be imported, because a foreign key must point to a row that exists. Rows are only removed and not split, so movie_award_id stays unique.

In [ ]:
award_movie = award_movie.dropna(subset=["movie_id"])
award_movie["movie_id"] = award_movie["movie_id"].astype(int)

print("Movie awards left:", len(award_movie))
print("Check, movie_award_id is unique:", award_movie["movie_award_id"].is_unique)

## STEP 13: keep only the ERD columns and save
pgAdmin puts the columns in the table by their place, so the order must be the same as in award_movie. The helper keys are not in the ERD, so they are left out. winner is removed because every row is a winner, and year_film because it is always year_ceremony minus 1. Storing the same information twice goes against normalization.

In [ ]:
award_movie = award_movie[["movie_award_id", "movie_id", "year_ceremony", "ceremony",
                           "category", "name_nominee", "film_title"]]

new_file = os.path.join(clean_folder, "award_movie_final.csv")
award_movie.to_csv(new_file, index=False)

print("Saved:", new_file)
print("File exists:", os.path.exists(new_file))

## STEP 14: check the saved file
We read the file back to check that it fits the table in PostgreSQL: the same number of rows, the columns in the right order, and no empty foreign keys. If one of these checks is False, the import in pgAdmin would fail or put values in the wrong column.

In [ ]:
saved = pd.read_csv(new_file)

print(saved.head())
print(saved.dtypes)
print("Check, same number of rows:", len(saved) == len(award_movie))
print("Check, primary key is first:", saved.columns[0] == "movie_award_id")
print("Check, no empty foreign keys:", saved["movie_id"].isna().sum() == 0)